In [1]:
import lava
print("Lava is available!")
print(lava.__file__)

Lava is available!
None


In [2]:
from lava.proc.lif.process import LIF
print("LIF imported successfully!")

LIF imported successfully!


In [3]:
import numpy as np
from lava.proc.lif.process import LIF
from lava.proc.dense.process import LearningDense
from lava.proc.learning_rules.r_stdp_learning_rule import RewardModulatedSTDP
from lava.magma.core.run_configs import Loihi2SimCfg
from lava.magma.core.run_conditions import RunSteps

print("Imports successful - checking what's actually available for STDP...")

Imports successful - checking what's actually available for STDP...


In [4]:
import lava.proc.learning_rules as lr
print([x for x in dir(lr) if not x.startswith('_')])

['r_stdp_learning_rule']


In [5]:
from lava.magma.core.learning.learning_rule import LoihiLearningRule
print("LoihiLearningRule found!")
help(LoihiLearningRule.__init__)

LoihiLearningRule found!
Help on function __init__ in module lava.magma.core.learning.learning_rule:

__init__(self, dw: Optional[str] = None, dd: Optional[str] = None, dt: Optional[str] = None, x1_impulse: Optional[float] = 0.0, x1_tau: Optional[float] = 0.0, x2_impulse: Optional[float] = 0.0, x2_tau: Optional[float] = 0.0, y1_impulse: Optional[float] = 0.0, y1_tau: Optional[float] = 0.0, y2_impulse: Optional[float] = 0.0, y2_tau: Optional[float] = 0.0, y3_impulse: Optional[float] = 0.0, y3_tau: Optional[float] = 0.0, t_epoch: Optional[int] = 1, rng_seed: Optional[int] = None) -> None
    Initialize self.  See help(type(self)) for accurate signature.



In [6]:
from lava.magma.core.learning.learning_rule import LoihiLearningRule
from lava.proc.dense.process import LearningDense
from lava.proc.io.sink import RingBuffer as Sink
from lava.proc.monitor.process import Monitor

# Define STDP rule using Lava's string-based syntax
# x1 = pre-synaptic trace, y1 = post-synaptic trace
# dw says: weight changes based on coincidence of trace and spike

learning_rule = LoihiLearningRule(
    dw='2*x1*y0 - 2*y1*x0',  # weight update: strengthen if pre-then-post, weaken if post-then-pre
    x1_impulse=16,
    x1_tau=10,
    y1_impulse=16, 
    y1_tau=10,
    t_epoch=1
)

print("STDP learning rule created!")
print(f"Weight update rule: {learning_rule.dw}")

STDP learning rule created!
Weight update rule: ProductSeries: decimate_exponent=None
    Product: dependency=y0, scaling=2*2^(7-7), target=dw, num_factors=1
        Factors:
            (x1+C, C = None)
    Product: dependency=x0, scaling=-2*2^(7-7), target=dw, num_factors=1
        Factors:
            (y1+C, C = None)



In [ ]:
# Two neurons connected via a learning synapse
input_neuron = LIF(shape=(1,), vth=10, dv=0, du=0, bias_mant=5)
output_neuron = LIF(shape=(1,), vth=10, dv=0, du=0.5, bias_mant=0)

# LearningDense uses the STDP rule we defined
initial_weight = np.array([[5]])

learning_dense = LearningDense(
    weights=initial_weight,
    learning_rule=learning_rule
)

input_neuron.s_out.connect(learning_dense.s_in)
learning_dense.a_out.connect(output_neuron.a_in)

# Connect output spikes back for learning (STDP needs post-synaptic feedback)
output_neuron.s_out.connect(learning_dense.s_in_bap)

# Monitor the weight over time
monitor = Monitor()
monitor.probe(learning_dense.weights, 100)

input_neuron.run(condition=RunSteps(num_steps=100), 
                 run_cfg=Loihi2SimCfg())

data = monitor.get_data()
input_neuron.stop()

print(data.keys())
print("Cell finished executing")

In [ ]:
import numpy as np
from lava.proc.lif.process import LIF
from lava.proc.dense.process import LearningDense
from lava.proc.io.sink import RingBuffer as Sink
from lava.proc.monitor.process import Monitor
from lava.magma.core.learning.learning_rule import LoihiLearningRule
from lava.magma.core.run_configs import Loihi2SimCfg
from lava.magma.core.run_conditions import RunSteps

print("Imports successful")

In [ ]:
learning_rule = LoihiLearningRule(
    dw='2*x1*y0 - 2*y1*x0',
    x1_impulse=16,
    x1_tau=10,
    y1_impulse=16, 
    y1_tau=10,
    t_epoch=1
)

print("Learning rule created")

In [ ]:
input_neuron = LIF(shape=(1,), vth=10, dv=0, du=0, bias_mant=5)
output_neuron = LIF(shape=(1,), vth=10, dv=0, du=0.5, bias_mant=0)

initial_weight = np.array([[5]])

learning_dense = LearningDense(
    weights=initial_weight,
    learning_rule=learning_rule
)

input_neuron.s_out.connect(learning_dense.s_in)
learning_dense.a_out.connect(output_neuron.a_in)

print("Network wired (without bap feedback yet)")

In [ ]:
monitor = Monitor()
monitor.probe(learning_dense.weights, 50)

input_neuron.run(condition=RunSteps(num_steps=50), 
                 run_cfg=Loihi2SimCfg())

data = monitor.get_data()
input_neuron.stop()

print("Run completed!")
print(data.keys())

In [ ]:
weight_trace = data['Process_2']['weights'].flatten()
print("Weight trace:")
print(weight_trace)
print(f"\nInitial weight: {weight_trace[0]}")
print(f"Final weight: {weight_trace[-1]}")
print(f"Did weight change? {weight_trace[0] != weight_trace[-1]}")

In [ ]:
print([p for p in dir(learning_dense) if 's_in' in p or 'bap' in p])

In [ ]:
input_neuron2 = LIF(shape=(1,), vth=10, dv=0, du=0, bias_mant=5)
output_neuron2 = LIF(shape=(1,), vth=10, dv=0, du=0.5, bias_mant=0)

learning_rule2 = LoihiLearningRule(
    dw='2*x1*y0 - 2*y1*x0',
    x1_impulse=16,
    x1_tau=10,
    y1_impulse=16, 
    y1_tau=10,
    t_epoch=1
)

initial_weight2 = np.array([[5]])
learning_dense2 = LearningDense(weights=initial_weight2, learning_rule=learning_rule2)

input_neuron2.s_out.connect(learning_dense2.s_in)
learning_dense2.a_out.connect(output_neuron2.a_in)
output_neuron2.s_out.connect(learning_dense2.s_in_bap)

monitor2 = Monitor()
monitor2.probe(learning_dense2.weights, 30)  # short run, just to test

print("About to run - if this hangs, interrupt after ~15 seconds")
input_neuron2.run(condition=RunSteps(num_steps=30), 
                  run_cfg=Loihi2SimCfg())
print("Completed without hanging!")

In [ ]:
import numpy as np
from lava.proc.lif.process import LIF
from lava.proc.dense.process import LearningDense
from lava.proc.monitor.process import Monitor
from lava.magma.core.learning.learning_rule import LoihiLearningRule
from lava.magma.core.run_configs import Loihi2SimCfg
from lava.magma.core.run_conditions import RunSteps

input_neuron3 = LIF(shape=(1,), vth=10, dv=0, du=0, bias_mant=5)
output_neuron3 = LIF(shape=(1,), vth=10, dv=0, du=0.5, bias_mant=0)

learning_rule3 = LoihiLearningRule(
    dw='2*x1*y0 - 2*y1*x0',
    x1_impulse=16, x1_tau=10,
    y1_impulse=16, y1_tau=10,
    t_epoch=1
)

initial_weight3 = np.array([[5]])
learning_dense3 = LearningDense(weights=initial_weight3, learning_rule=learning_rule3)

input_neuron3.s_out.connect(learning_dense3.s_in)
learning_dense3.a_out.connect(output_neuron3.a_in)
output_neuron3.s_out.connect(learning_dense3.s_in_bap)

monitor3 = Monitor()
monitor3.probe(learning_dense3.weights, 30)

input_neuron3.run(condition=RunSteps(num_steps=30), run_cfg=Loihi2SimCfg())

weight_data = monitor3.get_data()
input_neuron3.stop()

w_trace = list(weight_data.values())

## Lava STDP Implementation — Observations & Results

### Goal
Implement spike-timing-dependent plasticity (STDP) in Lava, equivalent to 
the working STDP simulation built in Brian2 (notebook 02), using Lava's 
`LoihiLearningRule` and `LearningDense` classes.

### Approach
Defined a Hebbian STDP rule using Lava's string-based syntax: dw = '2x1y0 - 2y1x0'

This strengthens weights when a post-synaptic spike (`y0`) follows a 
pre-synaptic trace (`x1`), and weakens them when a pre-synaptic spike 
(`x0`) follows a post-synaptic trace (`y1`) — directly equivalent to the 
`on_pre`/`on_post` STDP rules implemented in Brian2.

### Finding 1: Feedforward connection alone does not enable learning
Connecting only `input_neuron → learning_dense → output_neuron` (without 
feeding the output neuron's spikes back into the synapse) resulted in zero 
weight change over 50 timesteps:

| Step | Weight |
|------|--------|
| 0 | 5.0 |
| ... | 5.0 (constant) |
| 49 | 5.0 |

**Conclusion:** STDP requires explicit post-synaptic spike feedback via 
the `s_in_bap` (back-propagating action potential) port. Without it, the 
learning rule has no information about when the post-synaptic neuron fired, 
so no weight update can occur — confirming `s_in_bap` is functionally 
necessary, not optional, for this learning rule.

### Finding 2: Adding s_in_bap feedback causes a reproducible deadlock
Connecting `output_neuron.s_out → learning_dense.s_in_bap` (to provide the 
necessary post-synaptic feedback) caused the simulation to hang indefinitely 
— confirmed reproducibly across multiple attempts, with run times exceeding 
120 seconds and no completion.

### Root cause investigation
Searched Lava's official documentation and found this is a **known, 
documented limitation**. Lava's FAQ explicitly states: deadlocks occur 
"while designing recurrent processes" when a process waits for a signal at 
its input port while downstream activity depends on it — exactly the 
circular dependency created by feeding the output neuron's spike back into 
the same synapse driving it.

This is referenced as a known open issue in Lava's own GitHub repository 
(issues #32 and #100), not a configuration mistake on my part.

### Significance
This investigation demonstrates a genuine current limitation of Lava's 
process-based architecture for recurrent learning circuits — closing the 
loop between a neuron's output and a synapse's learning input creates a 
scheduling deadlock that the current runtime cannot resolve automatically. 
Real implementations on Loihi 2 hardware itself likely avoid this through 
dedicated on-chip learning engine pathways not fully exposed in software 
simulation, since hardware learning rules execute differently than the 
multiprocess software simulation used here.

**Comparison to Brian2:** Brian2's `event-driven` STDP equations handled 
this same logical dependency natively without any deadlock, since Brian2 
processes spike events sequentially within a single-process simulation 
rather than Lava's multiprocess architecture, which introduces real 
inter-process communication and scheduling constraints.

### Takeaway
Not every framework limitation is a personal mistake — sometimes the 
correct research outcome is identifying and documenting a genuine 
constraint, rather than forcing a workaround. This is itself a useful 
finding when evaluating Lava for STDP-based research projects.